# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
# I chose option 3: What is Noise?, by Alex Ross (Web)
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load()
document_text = "\n".join(d.page_content for d in docs)
print(len(document_text))
print(document_text[:200])

35406
What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th Anniversary


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
from openai import OpenAI
import os

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

In [13]:
from typing import Literal
from pydantic import BaseModel, Field

ToneType = Literal["Formal Academic Writing"]

class ArticleOut(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(..., description="a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development. ")
    Summary: str = Field(..., description="a concise and succinct summary no longer than 1000 tokens.")
    Tone: ToneType
    InputTokens: int = Field(..., ge=0)
    OutputTokens: int = Field(..., ge=0)

SCHEMA = ArticleOut.model_json_schema()

In [14]:
instructions = """
You are an helpful assistant that specializes in summary of documents.
Write the Summary in Formal Academic Writing style.
Relevance must be a a statement, no longer than one paragraph, 
that explains why is this article relevant for an AI professional in their professional development.
Summary must be concise and no longer than 1000 tokens.
Do not fabricate facts not supported by the provided context.
""".strip()

In [21]:
import json

PROMPT = """
Read this document:
{document}

Return Only valid JSON object with fields:
Author, Title, Relevance, Summary, Tone

Constraints:
- Tone must be "Formal Academic Writing"
- Relevance must be one paragraph
- Summary must be <= 1000 tokens
""".strip()

In [22]:
resp1 = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[{"role": "user", "content": PROMPT.format(document=document_text, length=length, input_tokens=0, output_tokens=0)}],
    temperature=0.2
)

in_tok = resp1.usage.input_tokens
out_tok = resp1.usage.output_tokens

In [ ]:
resp = client.responses.create(  
    model="gpt-4o-mini",
    instructions=instructions,
    input=[{"role": "user", "content": PROMPT.format(document=document_text)}],
    temperature=0.5
)

text = (resp.output_text or "").strip()              
start = text.find("{")                               
end = text.rfind("}")                                
data = json.loads(text[start:end+1])      
obj = ArticleOut.model_validate({
    **data,
    "Tone": "Formal Academic Writing",          
    "InputTokens": resp.usage.input_tokens,
    "OutputTokens": resp.usage.output_tokens
})


print(obj.model_dump_json)

<bound method BaseModel.model_dump_json of ArticleOut(Author='Alex Ross', Title='What Is Noise?', Relevance="This article is particularly relevant for AI professionals as it explores the multifaceted concept of noise, which has implications for data processing, signal transmission, and machine learning algorithms. Understanding the nature of noise, both in acoustic and informational contexts, can enhance AI practitioners' ability to develop systems that effectively manage and interpret data amidst the overwhelming 'noise' of modern information environments. Additionally, the article's examination of noise's cultural and social dimensions encourages a more holistic approach to AI development, emphasizing the importance of ethical considerations in technology.", Summary="In 'What Is Noise?', Alex Ross delves into the complex and often contradictory meanings of noise, tracing its evolution from a term denoting disturbance to a broader concept encompassing both auditory and informational c

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [28]:
%pip install -qU deepeval


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import os

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric, GEval

In [30]:
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

In [34]:
source_text = document_text          
summary_text = obj.Summary           

test_case = LLMTestCase(
    input=source_text,
    actual_output=summary_text
)

summarization_questions = [
    "Does the summary state the article’s central topic?",
    "Does it cover the main arguments and key conclusions?",
    "Does it omit information that is essential to understanding the conclusion?",
    "Does it add any facts not stated in the source text?",
    "Would a reader understand the main takeaway of the article from the summary alone?",
    ]

summ_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model=eval_model,
    include_reason=True
)

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Does it have a clear structure?",
        "Does the sentences connect smoothly?",
        "Are there unclear sentences that confuse the reader?",
        "Are there any obvious logical jumps or contradictions?",
        "Is the summary easy to follow for a general educated reader.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Does it maintains a formal academic tone?",
        "Does it avoids casual/slang expressions?",
        "Is the wording objective and non-emotional?",
        "Does it use appropriate connection words such as therefore, however, and additionally?",
        "Does it avoid exaggeration and absolute claims?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does it contains hate, harassment, or discriminatory content?",
        "Does contains sexual exploitation or other alarming content?",
        "Does it includes instructions for violence, illegal acts, or self-harm?",
        "Does the summary leaks personal data such as phone numbers, emails, addresses?",
        "Does it contain medical/legal/financial advice presented as certain facts?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

summ_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

results


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.4375,
 'SummarizationReason': 'The score is 0.44 because the summary includes numerous pieces of extra information that are not present in the original text, which indicates a lack of fidelity to the source material. This divergence from the original content significantly undermines the quality of the summarization.',
 'CoherenceScore': 0.85,
 'CoherenceReason': "The response has a clear structure, effectively outlining the main themes of Alex Ross's article. Sentences connect smoothly, providing a coherent flow of ideas. However, some complex phrases may challenge general readers, and while the summary is mostly easy to follow, a few sentences could be simplified for clarity. There are no obvious logical jumps or contradictions present.",
 'TonalityScore': 0.9005992634567705,
 'TonalityReason': "The response maintains a formal academic tone throughout, avoiding casual or slang expressions. The wording is objective and non-emotional, presenting a balanced analy

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [35]:
Better_PROMPT = """
You will rewrite the summary to be strictly faithful to the source.

SOURCE (use ONLY this):
{context}

PREVIOUS SUMMARY:
{old_summary}

EVALUATION FEEDBACK (what to fix):
{eval_reason}

Task:
Rewrite the summary in "Formal Academic Writing" style, and follow these rules:
1) Do NOT add any facts, examples, interpretations, or claims that cannot be directly supported by the source.
2) If you are not sure a detail is in the source, OMIT it.
3) Keep only the central thesis + the most important supporting points that are clearly present in the source.
4) Output the summary only.
""".strip()


In [36]:
new_resp = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,  
    input=[{
        "role": "user",
        "content": Better_PROMPT.format(
            context=document_text,
            old_summary=obj.Summary,                 
            eval_reason=results["SummarizationReason"]  
        )
    }],
    temperature=0.2
)

new_summary = (new_resp.output_text or "").strip()
print(new_summary[:800])

In "What Is Noise?", Alex Ross explores the multifaceted concept of noise, tracing its evolution from a term associated with disturbance to a broader understanding that encompasses both auditory and informational contexts. The article examines the historical and cultural significance of noise, highlighting its dual nature as both an unwanted sound and a source of creative expression. Ross discusses the ethical implications of noise, particularly in relation to social power dynamics and the subjective nature of sound perception. He also addresses the challenges of noise control and the impact of technological advancements on noise levels, ultimately framing noise as a pervasive condition that influences contemporary life. This exploration invites critical engagement with the auditory landsc


In [37]:
new_test_case = LLMTestCase(
    input=document_text,
    actual_output=new_summary
)

summ_metric.measure(new_test_case)
coherence_metric.measure(new_test_case)
tonality_metric.measure(new_test_case)
safety_metric.measure(new_test_case)

new_results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

new_results


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.7,
 'SummarizationReason': "The score is 0.70 because the summary introduces several pieces of extra information that are not present in the original text, such as references to Alex Ross and ethical implications of noise, which could mislead the reader about the original content's focus.",
 'CoherenceScore': 0.8622459331201855,
 'CoherenceReason': "The response has a clear structure, effectively outlining the main themes of Alex Ross's exploration of noise. The sentences connect smoothly, providing a coherent flow of ideas. There are no unclear sentences, and the summary avoids logical jumps or contradictions. It is accessible to a general educated reader, making it easy to follow. However, a minor improvement could be made by providing more explicit connections between the themes discussed.",
 'TonalityScore': 0.9237972961595118,
 'TonalityReason': "The response maintains a formal academic tone throughout and avoids casual or slang expressions. The wording is

Please, do not forget to add your comments.

In [38]:
# Comments
# The new summary shows a clear improvement: 
# the SummarizationScore increases from 0.44 to 0.70, 
# while coherence, tonality, and safety scores remain high. 
# This suggests the rewrite prompt successfully reduced hallucinated content 
# by pushing the model to be more source-faithful.
# However, there's still a room for further improvement.
# Also, it is possible that the evaluator misjudged because there might be a lot of noises on a webpage.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
